![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--anchor-browser-agent--anchor_browser_data_collection_guide)

# Anchor Browser Agent Guide: Data Collection (Grafana Dashboard)

A comprehensive, beginner-friendly guide for integrating Anchor Browser Agent into production-ready AI agents, focusing on the data collection use case.


## Introduction

Anchor Browser Agent is a cloud-based browser automation platform that enables AI agents to interact with web applications seamlessly. This guide demonstrates how to integrate Anchor into production-ready AI agents for the data collection use case.

## Key Benefits
- **Cloud-based:** No local browser installation required
- **AI-powered:** [Built-in AI agents](https://docs.anchorbrowser.io/agentic-browser-control/ai-task-completion?utm_source=agents-towards-production) for intelligent web interactions
- **Session Management:** [Rich browser session configurations](https://docs.anchorbrowser.io/api-reference/browser-sessions/start-browser-session?utm_source=agents-towards-production) for powerful session control
- **Proxy Support:** [Residential/mobile proxies](https://docs.anchorbrowser.io/advanced/proxy?utm_source=agents-towards-production) for Geographic distribution and IP rotation
- **Recording:** [Session recording](https://docs.anchorbrowser.io/essentials/recording?utm_source=agents-towards-production) for debugging and compliance
- **Profiles:** [Browser profile management](https://docs.anchorbrowser.io/essentials/authentication-and-identity?utm_source=agents-towards-production) for consistent user experiences


## Visual Overview

![Data Collection Workflow](./assets/data-collection-diagram.png)

## Requirements
 - Python 3.x or Node.js (Full code in separated files)

## Getting Started

### 1. Sign Up and Get Your API Key
- Go to [anchorbrowser.io](https://anchorbrowser.io?utm_source=agents-towards-production) and sign up:
<img src="./assets/signup.png" alt="Signup" width="200"/>

- Visit the [API Access Page](https://app.anchorbrowser.io/api-access?utm_source=agents-towards-production) to generate your API key:
<img src="./assets/dashboard.png" alt="API Key" width="400"/>

### 2. Install Dependencies
- **Node.js:**
    ```bash
    npm install playwright-core axios
    ```
- **Python:**

In [ ]:
!pip3 install playwright requests

## Use Case: Data Collection (Grafana Dashboard)

### Scenario
An AI agent collects data from a Grafana dashboard. The agent extracts structured data from the dashboard.
If authentication is required, A browser profile can be used. (authenticate once).

**Grafana dashboard example:**
<img src="./assets/grafana-dashboard.png" alt="Grafana Dashboard" width="400"/>

### Extract Data


In [ ]:
from playwright.async_api import async_playwright

ANCHOR_API_KEY = "<ANCHOR-API-KEY>"   # Insert your ANCHOR-API-KEY.
connection_string = f"wss://connect.anchorbrowser.io?apiKey={ANCHOR_API_KEY}"

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(connection_string)
    context = browser.contexts[0]
    ai = context.service_workers[0]
    page = context.pages[0]

    await page.goto(
        "https://play.grafana.org/a/grafana-k8s-app/navigation/nodes?from=now-1h&to=now&refresh=1m",
        wait_until="domcontentloaded"
    )
    result = await ai.evaluate('Collect the node names and their CPU average %, return in JSON array')
    print(result)

**Note:** This code uses **async_playwright** to be compatible with Jupyter notebook's asyncio event loop. For regular Python scripts, you can use **sync_playwright** instead. (Don't forget to exclude await keywords as well)

## Best Practices

- Implement retry logic and logging
- Use [session timeouts](https://docs.anchorbrowser.io/advanced/session-termination?utm_source=agents-towards-production) and clean up sessions
- Use [persistent profiles](https://docs.anchorbrowser.io/essentials/authentication-and-identity?utm_source=agents-towards-production) for authenticated sessions
- Batch related tasks in single sessions
- Use environment variables for API keys
- Enable [session recording](https://docs.anchorbrowser.io/essentials/recording?utm_source=agents-towards-production) for debugging


## Advanced: Browser Profile
### 1. Create a Browser Profile 
Browser Profile is a 
In order to use a [browser profile](https://docs.anchorbrowser.io/essentials/authentication-and-identity?utm_source=agents-towards-production) it should be created first in one of 3 ways:
 1. Create a new profile using the [Browser Playground](https://app.anchorbrowser.io/playground?utm_source=agents-towards-production), Save Profile **after authentication**.
<img src="./assets/configure-profile.png" alt="API Key" width="600"/>

 2. Create a new profile when [creating a session using the API](https://docs.anchorbrowser.io/api-reference/browser-sessions/start-browser-session?utm_source=agents-towards-production). The important configurations are:
    ```json
    {
        "browser": {
            "profile": {
                "name": "name-for-profile",
                "persist": true
            }
        }
    }
    ```
    
   
 3. Create a profile using the [designated API](https://docs.anchorbrowser.io/api-reference/profiles/create-profile?utm_source=agents-towards-production). 

**Important** - Either way the profile should be **used to login** in order to use authentication in later sessions.

For more information about browser profiles [see our Docs](https://docs.anchorbrowser.io/essentials/authentication-and-identity?utm_source=agents-towards-production)


#### 2. Load a Browser Profile
Usage of existing profiles is similar to Profile Creation (see above). Here is a simple code implementation for the API usage:
##### First Create a session using the profile you wish to use
**note:** If you are not sure which profiles you have try our [list profiles API](https://docs.anchorbrowser.io/api-reference/profiles/list-profiles?utm_source=agents-towards-production) to view all your existing profiles first.


In [ ]:
import requests

url = "https://api.anchorbrowser.io/v1/sessions"

payload = {
    "browser": {
        "profile": {
            "name": "my-profile",                              # Replace "my-profile" to match your existing profile name
            "persist": True
        }
    },
    "session": {
        "timeout": {                                           # Reduced timeouts for cost-effective learning
            "max_duration": 4,
            "idle_timeout": 2
        }
    }
}
headers = {
    "anchor-api-key": "<ANCHOR-API-KEY>",   # Set your ANCHOR-API-KEY.
    "Content-Type": "application/json"
}

# Initialize session using profile "my-profile"
response = requests.request("POST", url, json=payload, headers=headers)

# Make sure to keep the cdpUrl for the next step
session_data = response.json()
print(session_data)

connection_string = session_data['data']['cdp_url']

##### Then use the sessionId in the connection string

In [ ]:
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.connect_over_cdp(connection_string)   # Connection string format: wss://connect.anchorbrowser.io?apiKey={api-key}&sessionId={sessionId}
    context = browser.contexts[0]
    ai = context.service_workers[0]
    page = context.pages[0]

    await page.goto(
        "https://play.grafana.org/a/grafana-k8s-app/navigation/nodes?from=now-1h&to=now&refresh=1m",
        wait_until="domcontentloaded"
    )
    result = await ai.evaluate("Collect the names and their CPU average %, return in JSON array")
    print(result)

## Troubleshooting

**Common Issues:**
- Session connection failures: Check API key and session ID
- AI agent failures: Adjust prompt. Also maybe try to run again.
- Session timeout: Use a new session with increased timeout.

**Debugging Tips:**
- Enable [session recording](https://docs.anchorbrowser.io/essentials/recording?utm_source=agents-towards-production)
- Use live view for real-time monitoring (Link in session data or in [Session History Page](https://app.anchorbrowser.io/sessions?utm_source=agents-towards-production) while the session still running)


## Resources & Support

- [Anchor Browser Agent Documentation](https://docs.anchorbrowser.io?utm_source=agents-towards-production)
- [API Reference](https://docs.anchorbrowser.io/api?utm_source=agents-towards-production)
- Email: [support@anchorbrowser.io](mailto:support@anchorbrowser.io?utm_source=agents-towards-production)
